# Colab gpu kernel playground

This notebook writes a tiny CUDA kernel, compiles it, and runs it on the Colab GPU.


In [4]:
%%writefile custom_kernel.cu
#include <cstdlib>
#include <cmath>
#include <iostream>
#include <cuda_runtime.h>
#include <cublas_v2.h>

// A(m, n) x B(n, k)
#define TILE_WIDTH 16
__global__ void tiledMatMul(float* A, float* B, float* C, int m, int n, int k) {
    __shared__ float Ads[TILE_WIDTH][TILE_WIDTH];
    __shared__ float Bds[TILE_WIDTH][TILE_WIDTH];

    int bx = blockIdx.x;
    int by = blockIdx.y;
    int tx = threadIdx.x;
    int ty = threadIdx.y;

    // output elem we're working on
    int Row = by * TILE_WIDTH + ty;
    int Col = bx * TILE_WIDTH + tx;

    // loop over required tiles
    float Cvalue = 0.0f;
    for (int ph = 0; ph < ceil(n / (float)TILE_WIDTH); ++ph) {
        // collaboartive effort to load values into shared memory from each thread
        if ((Row < m) && (ph * TILE_WIDTH + tx < n)) {
            Ads[ty][tx] = A[Row * n + ph * TILE_WIDTH + tx];
        } else {
            Ads[ty][tx] = 0.0f;
        }

        if ((ph * TILE_WIDTH + ty < n) && (Col < k)) {
            Bds[ty][tx] = B[(ph * TILE_WIDTH + ty) * k + Col];
        } else {
            Bds[ty][tx] = 0.0f;
        }
        __syncthreads();

        for (int i = 0; i < TILE_WIDTH; ++i) {
            Cvalue += Ads[ty][i] * Bds[i][tx];
        }

        __syncthreads();
    }

    if ((Row < m) && (Col < k)) {
        C[Row * k + Col] = Cvalue;
    }
}

void checkCuda(cudaError_t result, const char* message) {
    if (result != cudaSuccess) {
        std::cerr << message << ": " << cudaGetErrorString(result) << std::endl;
        std::exit(1);
    }
}

void checkCublas(cublasStatus_t result, const char* message) {
    if (result != CUBLAS_STATUS_SUCCESS) {
        std::cerr << message << std::endl;
        std::exit(1);
    }
}

int main() {
    const int m = 37;
    const int n = 29;
    const int k = 41;

    const int aSize = m * n;
    const int bSize = n * k;
    const int cSize = m * k;

    float* hA = new float[aSize];
    float* hB = new float[bSize];
    float* hKernelC = new float[cSize];
    float* hCublasC = new float[cSize];

    std::srand(1234);
    for (int i = 0; i < aSize; ++i) {
        hA[i] = static_cast<float>((std::rand() % 21) - 10);
    }
    for (int i = 0; i < bSize; ++i) {
        hB[i] = static_cast<float>((std::rand() % 21) - 10);
    }

    float* dA;
    float* dB;
    float* dKernelC;
    float* dCublasC;

    checkCuda(cudaMalloc((void**)&dA, aSize * sizeof(float)), "cudaMalloc dA failed");
    checkCuda(cudaMalloc((void**)&dB, bSize * sizeof(float)), "cudaMalloc dB failed");
    checkCuda(cudaMalloc((void**)&dKernelC, cSize * sizeof(float)), "cudaMalloc dKernelC failed");
    checkCuda(cudaMalloc((void**)&dCublasC, cSize * sizeof(float)), "cudaMalloc dCublasC failed");

    checkCuda(cudaMemcpy(dA, hA, aSize * sizeof(float), cudaMemcpyHostToDevice),
              "cudaMemcpy hA -> dA failed");
    checkCuda(cudaMemcpy(dB, hB, bSize * sizeof(float), cudaMemcpyHostToDevice),
              "cudaMemcpy hB -> dB failed");

    dim3 dimBlock(TILE_WIDTH, TILE_WIDTH);
    dim3 dimGrid((k + TILE_WIDTH - 1) / TILE_WIDTH,
                 (m + TILE_WIDTH - 1) / TILE_WIDTH);

    tiledMatMul<<<dimGrid, dimBlock>>>(dA, dB, dKernelC, m, n, k);
    checkCuda(cudaGetLastError(), "kernel launch failed");
    checkCuda(cudaDeviceSynchronize(), "kernel execution failed");

    cublasHandle_t handle;
    checkCublas(cublasCreate(&handle), "cublasCreate failed");

    const float alpha = 1.0f;
    const float beta = 0.0f;

    // cuBLAS assumes column-major matrices, so compute C^T = B^T * A^T.
    checkCublas(
        cublasSgemm(handle,
                    CUBLAS_OP_N,
                    CUBLAS_OP_N,
                    k,
                    m,
                    n,
                    &alpha,
                    dB,
                    k,
                    dA,
                    n,
                    &beta,
                    dCublasC,
                    k),
        "cublasSgemm failed");

    checkCuda(cudaMemcpy(hKernelC, dKernelC, cSize * sizeof(float), cudaMemcpyDeviceToHost),
              "cudaMemcpy dKernelC -> hKernelC failed");
    checkCuda(cudaMemcpy(hCublasC, dCublasC, cSize * sizeof(float), cudaMemcpyDeviceToHost),
              "cudaMemcpy dCublasC -> hCublasC failed");

    bool matches = true;
    float maxAbsDiff = 0.0f;
    int mismatchIndex = -1;

    for (int i = 0; i < cSize; ++i) {
        float diff = std::fabs(hKernelC[i] - hCublasC[i]);
        if (diff > maxAbsDiff) {
            maxAbsDiff = diff;
        }
        if (diff > 1e-3f && mismatchIndex == -1) {
            matches = false;
            mismatchIndex = i;
        }
    }

    std::cout << "A shape: " << m << " x " << n << std::endl;
    std::cout << "B shape: " << n << " x " << k << std::endl;
    std::cout << "C shape: " << m << " x " << k << std::endl;
    std::cout << "Max absolute difference: " << maxAbsDiff << std::endl;

    if (matches) {
        std::cout << "Check passed: kernel output matches cuBLAS." << std::endl;
    } else {
        std::cout << "Check failed: kernel output does not match cuBLAS." << std::endl;
        std::cout << "First mismatch index: " << mismatchIndex << std::endl;
        std::cout << "Kernel value: " << hKernelC[mismatchIndex] << std::endl;
        std::cout << "cuBLAS value: " << hCublasC[mismatchIndex] << std::endl;
    }

    checkCublas(cublasDestroy(handle), "cublasDestroy failed");
    checkCuda(cudaFree(dA), "cudaFree dA failed");
    checkCuda(cudaFree(dB), "cudaFree dB failed");
    checkCuda(cudaFree(dKernelC), "cudaFree dKernelC failed");
    checkCuda(cudaFree(dCublasC), "cudaFree dCublasC failed");

    delete[] hA;
    delete[] hB;
    delete[] hKernelC;
    delete[] hCublasC;

    return 0;
}

Overwriting custom_kernel.cu


In [5]:
!nvidia-smi

Mon May  4 23:55:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [9]:
!nvcc custom_kernel.cu -o custom_kernel -lcublas

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [10]:
!./custom_kernel

A shape: 37 x 29
B shape: 29 x 41
C shape: 37 x 41
Max absolute difference: 0
Check passed: kernel output matches cuBLAS.


In [5]:
!ncu ./custom_kernel

==PROF== Connected to process 16600 (/content/custom_kernel)
==PROF== Profiling "matMul" - 0: 0%....50%....100% - 10 passes
Matrix A (2x3):
1	2	3	
4	5	6	

Matrix B (3x4):
7	8	9	10	
11	12	13	14	
15	16	17	18	

Matrix C = A x B (2x4):
74	80	86	92	
173	188	203	218	
==PROF== Disconnected from process 16600
[16600] custom_kernel@127.0.0.1
  matMul(float *, float *, float *, int, int, int) (2, 1, 1)x(2, 2, 1), Context 1, Stream 7, Device 0, CC 8.0
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         1.19
    SM Frequency                    Ghz         1.08
    Elapsed Cycles                cycle        3,942
    Memory Throughput                 %         0.47
    DRAM Throughput                   %         0.11
    Duration                         us         3.65
    L1/TEX Cache Throughput        